Hi, playing around with whatsapp automation and scraping,
tried to send watsapp alerts to friend's number when the booking for Spiderman movie opens. 

P.S - Sorry, only focused on getting the job done, whilst making this script time constraints. 
</br> We can make impvoements like, parametrising the movie url, requested slot date, user number prompts, run it in background & may tie it up in streamlit UI--- stuff like that.

In [ ]:
import os
import selenium
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
import time
from PIL import Image
import io
import requests
from webdriver_manager.chrome import ChromeDriverManager
from selenium.common.exceptions import ElementClickInterceptedException
import pywhatkit

In [ ]:
#Install Driver
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)

In [ ]:
def get_it(): 
    search_url = "https://in.bookmyshow.com/movies/bengaluru/project-hail-mary/buytickets/ET00481564/20260403"
    driver.get(search_url)

    try:
        cls_box = WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.XPATH, "(//*[contains(@class, 'slick-track')])"))
        )
        x = cls_box.text.split("\n")
        print(x)
        return x
    except TimeoutException:
        print("Element not found - page may have changed or tickets not yet available")
        return []

def show_it(x):
    y = ""
    for i in range(0, len(x), 2):
        y += "\n" + "open for " + x[i] + " on " + x[i+1]
    return y

def send_msg(w, H, M):
    ur_number = 'place the phone number here'
    pywhatkit.sendwhatmsg(ur_number, w, H, M, 35, True, 2)

In [4]:
timestamp = time.strftime('%H:%M')
H, M = timestamp.split(":")
print(H, " ",M)

01   22


In [ ]:
while 1:
    r = get_it()
    if not r:
        print("Retrying in 30 mins...")
        time.sleep(1800)
        continue
    if '19' in r:
        h, m = time.strftime('%H:%M').split(":")
        send_msg("Booking Open...Let's GO", int(h), int(m)+3)
        break
    else:
        h, m = time.strftime('%H:%M').split(":")
        msg_recipe = show_it(r)
        send_msg(msg_recipe, int(h), int(m)+3)
        time.sleep(10800)

In [ ]:
# --- Test it ---
movie_url = "https://in.bookmyshow.com/movies/bengaluru/project-hail-mary/buytickets/ET00481564/20260403"
target_date = "3"  # April 3

available = is_booking_available(movie_url, target_date)
print(f"Booking available: {available}")

In [ ]:
def is_booking_available(movie_url, target_date):
    """
    Check if a movie is available for booking on a given date.

    Args:
        movie_url   : BookMyShow URL for the movie's buy tickets page
        target_date : Day of month as string, e.g. "03" or "3"

    Returns:
        True if the date is found in the date carousel, False otherwise
    """
    driver.get(movie_url)

    try:
        carousel = WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.XPATH, "(//*[contains(@class, 'slick-track')])"))
        )
        available_dates = carousel.text.split("\n")
        print(f"Available dates found: {available_dates}")
        return str(int(target_date)) in available_dates  # normalize "03" -> "3"

    except TimeoutException:
        print("Could not load date carousel — page structure may have changed or tickets not released yet")
        return False